# 02 — Certificar dados e consumir split_v2

## Objetivo
Confirmar que os dados utilizados correspondem ao inventário auditado e que os grupos permanecem independentes entre partições.

## Entradas
RAW, metadados oficiais, auditoria v2 e manifesto de `data/processed/split_v2/candidate/`.

## Saídas
`reports/experimental_v2/dataset_contract.json`: certificado com hashes e versão. Geração de outra proposta somente por configuração explícita.

## Permissões de dados
Custódia dos dados: pode verificar IDs, labels e hashes de todas as partições. Não avalia modelos nem inspeciona predições.

## Reprodutibilidade
Seeds 20260928/20260929. O algoritmo de split fica em `experimental_validity.py`; este notebook não implementa divisão aleatória própria.

Protocolo: [experimental_protocol_v2.md](../docs/experimental_protocol_v2.md).
Os resultados anteriores estão catalogados como `legacy_v1`; não comparar seus scores diretamente aos v2.


In [ ]:
from pathlib import Path
import sys

# Bootstrap de importação; a resolução dos dados fica nos módulos compartilhados.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/experimental_protocol.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scripts.v2_artifacts import repository_root, read_json, run_id
ROOT = repository_root()

from scripts.experimental_validity import official_metadata, audit, propose, validate_split
from scripts.v2_data import METADATA_REL, SPLIT_REL, AUDIT_REL, CONTRACT_REL, read_audit, certify_dataset
AUDIT_MODE = "verify_existing"  # ou "generate_new", com outro diretório
GENERATE_SPLIT = False
AUDIT_DIRECTORY = AUDIT_REL
SPLIT_DIRECTORY = SPLIT_REL
CONTRACT_PATH = CONTRACT_REL
SPLIT_SEED = 20260928


## 1. Associar imagem → lesão com schema estrito
Labels inválidos, IDs sem associação ou classes conflitantes em uma lesão interrompem o fluxo.
Não remover linhas silenciosamente. Não inferir pacientes pela aparência ou por números dos IDs.


In [ ]:
metadata = official_metadata(ROOT, ROOT / METADATA_REL)
display(metadata[["image_stem", "lesion_id", "diagnosis_confirm_type", "target_label"]].head())
print(f"{len(metadata):,} imagens; {metadata.lesion_id.nunique():,} lesões")


## 2. Reutilizar a auditoria ou gerar explicitamente outra
A opção padrão verifica os artefatos existentes. `generate_new` recalcula hashes e busca perceptual; pode levar alguns minutos.
Para gerar novamente, escolha outro `AUDIT_DIRECTORY`. O script recusa sobrescrita.
O mesmo vale para uma nova proposta: use outro `SPLIT_DIRECTORY` e outro certificado.
Não experimentar seeds de split para escolher a que produz melhores scores.


In [ ]:
if AUDIT_MODE == "generate_new":
    audit(ROOT, ROOT / METADATA_REL, ROOT / AUDIT_DIRECTORY, radius=6, workers=4)
elif AUDIT_MODE != "verify_existing":
    raise ValueError("AUDIT_MODE desconhecido")
inventory, audit_summary = read_audit(ROOT, AUDIT_DIRECTORY)
display(pd.Series(audit_summary["pair_categories"]).to_frame("pares"))
if GENERATE_SPLIT:
    propose(ROOT, ROOT / AUDIT_DIRECTORY, ROOT / SPLIT_DIRECTORY, SPLIT_SEED, precaution_radius=6)


## 3. Verificar a proposta completa
O certificado exige: associação ao suplemento, igualdade com o inventário auditado, grupos conectados corretos,
classes consistentes, partições iguais às atribuições e hashes atuais de todas as imagens/máscaras.
NB03–06 consumirão esse certificado e somente seus dados autorizados. Não precisam abrir o holdout para repetir a auditoria.


In [ ]:
certificate = certify_dataset(ROOT, SPLIT_DIRECTORY, AUDIT_DIRECTORY, METADATA_REL, CONTRACT_PATH)
display(pd.Series({k: certificate[k] for k in ["dataset", "dataset_version", "split_version", "grouping_unit",
                                               "split_seed", "calibration_seed", "certificate_id",
                                               "dataset_manifest_sha256", "metadata_sha256", "split_manifest_sha256"]}).to_frame("valor"))
display(pd.Series(certificate["integrity"]).to_frame("invariante"))


## 4. Distribuição e compromisso da reserva de calibração
A divisão base é 70/15/15. A reserva de calibração foi retirada antes de treinar: os papéis efetivos são aproximadamente 60/15/10/15.
Todas as vistas da mesma lesão ficam juntas. DF e VASC têm poucos grupos; a incerteza estatística não desaparece com mais vistas.


In [ ]:
distribution = pd.read_csv(ROOT / SPLIT_DIRECTORY / "distribution.csv")
for value in ["images", "groups", "class_proportion_in_split"]:
    display(distribution.pivot(index="class", columns="split", values=value).rename_axis(f"classe / {value}"))
ax = distribution.pivot(index="class", columns="split", values="groups").plot.bar(figsize=(11, 4))
ax.set(title="Grupos independentes por classe e partição", xlabel="Classe", ylabel="Lesões/grupos")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()


## Conclusão
Os conjuntos estão certificados por lesão e identidade de conteúdo. Independência por paciente não foi estabelecida. O holdout interno usa um corpus historicamente observado e não equivale a um final test independente.

## Artefatos produzidos
`dataset_contract.json` é imutável para esta versão. RAW, splits e resultados v1 permanecem intactos. Os nomes/hashs da proposta aparecem no resumo acima.

## Próxima etapa
NB03: baselines usando exclusivamente train e validation certificados.
